# Method Viability Analysis

This notebook evaluates geometry-aware adaptive sampling of primal and dual LP candidates as an active-set discovery method. The goal is to check whether the method finds useful recurring support structure, not whether it replaces HiGHS, simplex, or interior-point solvers.

## Viability Standard

A favorable result here means the sampler repeatedly identifies active-set support that agrees with a HiGHS reference, while feasibility, duality-gap, complementarity, and certificate checks keep invalid candidates from being counted as solutions.

A stronger solver-replacement claim would require reliable certified optimality and total end-to-end runtime wins. This notebook treats that as out of scope unless the evidence is already present.

In [1]:
from __future__ import annotations

import json
import math
import sys
from collections import defaultdict
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

candidate = Path.cwd().resolve()
for path in (candidate, *candidate.parents):
    if (path / "src" / "lpas").exists():
        ROOT = path
        break
else:
    raise RuntimeError("Could not locate repository root containing src/lpas")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RUN_FULL_BENCHMARKS = False
RUN_REDUCED_FULL_MATRIX = True
GENERATED_OUTPUT_ROOT = ROOT / "notebooks" / "method_viability_outputs"

display(Markdown(f"Repository root: `{ROOT}`"))

Repository root: `/Users/nickgrebe/Projects/LP_Experiment`

In [2]:
from lpas.core.primal_dual import evaluate_primal_dual_pairs
from lpas.core.scoring import effective_gap, score_candidates
from lpas.experiments.benchmark_runner import (
    GEOMETRY_AWARE_METHOD,
    GEOMETRY_AWARE_POLISHED_METHOD,
    NAIVE_MONTE_CARLO_METHOD,
    run_problem_comparison,
)
from lpas.experiments.certificate_validation import write_certificate_validation_outputs
from lpas.experiments.corner_discovery import run_corner_discovery_experiment, write_corner_discovery_outputs
from lpas.experiments.generators import tiny_known_lp
from lpas.experiments.scaling_by_dimension import run_scaling_experiment, write_scaling_outputs
from lpas.experiments.solver_seeding_total_time import (
    run_solver_seeding_total_time_benchmark,
    write_solver_seeding_outputs,
)
from lpas.problem_generation.random_feasible_lps import generate_random_feasible_bounded_lp
from lpas.solver.scipy_handoff import solve_with_scipy
from lpas.utils.config import SamplerConfig, SolverConfig

METHOD_LABELS = {
    GEOMETRY_AWARE_METHOD: "geometry-aware",
    GEOMETRY_AWARE_POLISHED_METHOD: "geometry-aware + polishing",
    NAIVE_MONTE_CARLO_METHOD: "naive Monte Carlo",
}

def jsonify(value: Any) -> Any:
    if is_dataclass(value):
        return {key: jsonify(item) for key, item in asdict(value).items()}
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): jsonify(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonify(item) for item in value]
    return value

def safe_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(number):
        return None
    return number

def mean(values: list[Any]) -> float | None:
    numeric = [safe_float(value) for value in values]
    numeric = [value for value in numeric if value is not None]
    if not numeric:
        return None
    return float(np.mean(numeric))

def fmt(value: Any, digits: int = 4) -> str:
    number = safe_float(value)
    if number is None:
        return "-"
    if abs(number) >= 1000 or (0 < abs(number) < 1e-3):
        return f"{number:.3e}"
    return f"{number:.{digits}f}"

def markdown_table(headers: list[str], rows: list[list[Any]]) -> Markdown:
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(item) for item in row) + " |")
    return Markdown("\n".join(lines))

def load_json_first(paths: list[Path]) -> tuple[Path | None, dict[str, Any] | None]:
    for path in paths:
        absolute = ROOT / path
        if absolute.exists():
            return absolute, json.loads(absolute.read_text(encoding="utf-8"))
    return None, None

def records_from_payload(payload: dict[str, Any] | None) -> list[dict[str, Any]]:
    if payload is None:
        return []
    records = payload.get("records")
    if isinstance(records, list):
        return [dict(row) for row in records]
    return []

def summarize_by_method(records: list[dict[str, Any]], metrics: list[str], method_key: str = "method") -> list[list[str]]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in records:
        grouped[str(row.get(method_key, "unknown"))].append(row)
    table_rows = []
    for method, rows in sorted(grouped.items()):
        table_rows.append([METHOD_LABELS.get(method, method), str(len(rows)), *[fmt(mean([row.get(metric) for row in rows])) for metric in metrics]])
    return table_rows

## Optional Full Regeneration

The default notebook path does not regenerate heavy artifacts. Set `RUN_FULL_BENCHMARKS = True` above to produce fresh outputs under `notebooks/method_viability_outputs/`. Keep `RUN_REDUCED_FULL_MATRIX = True` for a reduced matrix, or set it to `False` for the full dimensions used by `experiments/run_full_extension_evaluation.py`.

In [3]:
def build_viability_config(seed: int = 0, quick: bool = True) -> SolverConfig:
    iterations = 10 if quick else 24
    return SolverConfig(
        batch_size=96 if quick else 256,
        max_iter=iterations,
        patience=max(4, iterations // 2),
        seed=seed,
        sampler=SamplerConfig(seed=seed, sigma_init=1.35, primal_init_mean=0.8, dual_init_mean=0.8),
    )

if RUN_FULL_BENCHMARKS:
    config = build_viability_config(quick=RUN_REDUCED_FULL_MATRIX)
    dims = (5, 10, 15, 20) if RUN_REDUCED_FULL_MATRIX else (2, 5, 10, 12, 15, 18, 20, 30, 50)
    ratios = (2, 5) if RUN_REDUCED_FULL_MATRIX else (2, 5, 10)
    seed_range = range(2) if RUN_REDUCED_FULL_MATRIX else range(5)
    scaling_records = run_scaling_experiment(config=config, dimensions=dims, ratios=ratios, seeds=seed_range, backend="numpy_cpu")
    write_scaling_outputs(scaling_records, GENERATED_OUTPUT_ROOT, config_payload={"source": "notebook", "reduced": RUN_REDUCED_FULL_MATRIX})
    corner_records = run_corner_discovery_experiment(config=config, dimensions=dims, ratios=ratios, seeds=seed_range)
    write_corner_discovery_outputs(corner_records, GENERATED_OUTPUT_ROOT, config_payload={"source": "notebook", "reduced": RUN_REDUCED_FULL_MATRIX})
    seeding_dims = (2,) if RUN_REDUCED_FULL_MATRIX else (2, 3, 4)
    seeding_ratios = (2,) if RUN_REDUCED_FULL_MATRIX else (2, 4)
    seeding_records = run_solver_seeding_total_time_benchmark(config=config, dimensions=seeding_dims, ratios=seeding_ratios, seeds=seed_range, backend="numpy_cpu")
    write_solver_seeding_outputs(seeding_records, GENERATED_OUTPUT_ROOT, config_payload={"source": "notebook", "reduced": RUN_REDUCED_FULL_MATRIX})
    write_certificate_validation_outputs(GENERATED_OUTPUT_ROOT)
    display(Markdown(f"Generated fresh viability outputs under `{GENERATED_OUTPUT_ROOT}`."))
else:
    display(Markdown("Full benchmark regeneration skipped. The notebook will use a live smoke check plus checked-in artifacts where available."))

Full benchmark regeneration skipped. The notebook will use a live smoke check plus checked-in artifacts where available.

## Primal-Dual Diagnostics And Scoring

The score combines objective value, primal feasibility, dual feasibility, effective duality gap, complementarity, geometry support, and active-set recurrence. The important guardrail is that a high primal objective does not count as evidence if feasibility or dual certification fails.

In [4]:
problem = tiny_known_lp()
X = np.array(
    [
        [2.0, 2.0],  # optimum
        [3.0, 3.0],  # infeasible and too good
        [1.0, 1.0],  # primal feasible but dual partner is invalid
        [1.0, 1.0],  # feasible pair with loose complementarity
    ],
    dtype=float,
)
Y = np.array(
    [
        [2.0, 1.0, 0.0],
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0],
        [2.0, 1.0, 0.0],
    ],
    dtype=float,
)
labels = ["optimal pair", "too-good infeasible", "bad dual", "loose complementarity"]
metrics = evaluate_primal_dual_pairs(problem, X, Y)
scores = score_candidates(metrics)
rows = []
for i, label in enumerate(labels):
    rows.append(
        [
            label,
            fmt(metrics.primal_objective[i]),
            fmt(metrics.dual_objective[i]),
            fmt(metrics.raw_gap[i]),
            fmt(effective_gap(metrics)[i]),
            fmt(metrics.primal_violation_norm[i]),
            fmt(metrics.dual_violation_norm[i]),
            fmt(metrics.complementarity_error[i]),
            fmt(scores[i]),
        ]
    )
display(markdown_table(["candidate", "primal obj", "dual obj", "raw gap", "effective gap", "p viol", "d viol", "comp", "score"], rows))

| candidate | primal obj | dual obj | raw gap | effective gap | p viol | d viol | comp | score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| optimal pair | 10.0000 | 10.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 4.5000 |
| too-good infeasible | 15.0000 | 0.0000 | -15.0000 | 23.0000 | 3.0000 | 5.0000 | 0.0000 | 1.9667 |
| bad dual | 5.0000 | 0.0000 | -5.0000 | 10.0000 | 0.0000 | 5.0000 | 0.0000 | 4.0167 |
| loose complementarity | 5.0000 | 10.0000 | 5.0000 | 5.0000 | 0.0000 | 0.0000 | 5.0000 | 5.8167 |

The `too-good infeasible` row is intentionally attractive by raw primal objective. Its negative raw gap and primal/dual violations become a large effective gap penalty, so it is diagnostic evidence, not a valid solution.

## Live Matched-Budget Smoke Check

This small run is not the full evidence base. It verifies that the notebook can execute the repository APIs and that geometry-aware and naive candidates are compared under the same sample budget.

In [5]:
smoke_config = build_viability_config(seed=0, quick=True)
generated = generate_random_feasible_bounded_lp(n_variables=5, n_constraints=12, seed=0)
smoke_results = run_problem_comparison(
    problem_name=generated.name,
    family="method_viability_smoke",
    problem=generated.problem,
    config=smoke_config,
    capture_samples=False,
)

smoke_rows = []
for result in smoke_results:
    smoke_rows.append(
        [
            METHOD_LABELS.get(result.method, result.method),
            result.n_samples_total,
            fmt(result.active_set_recovery_accuracy),
            str(result.exact_active_set_match),
            fmt(result.objective_gap_to_highs),
            fmt(result.best_primal_violation),
            fmt(result.best_dual_violation),
            fmt(result.best_complementarity_error),
            fmt(result.best_certified_gap),
            fmt(result.wall_clock_seconds),
        ]
    )

display(markdown_table(
    ["method", "samples", "active Jaccard", "exact active set", "obj gap", "best p viol", "best d viol", "best comp", "cert gap", "seconds"],
    smoke_rows,
))

| method | samples | active Jaccard | exact active set | obj gap | best p viol | best d viol | best comp | cert gap | seconds |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| geometry-aware | 960 | 0.8000 | False | 0.9011 | 0.0000 | 0.0000 | 1.6179 | 8.2133 | 0.0120 |
| naive Monte Carlo | 960 | 0.8333 | False | 2.2134 | 0.0000 | 0.0000 | 1.4355 | 10.6486 | 0.0066 |
| geometry-aware + polishing | 960 | 0.8000 | False | 0.5438 | 0.0000 | 0.0000 | 1.6179 | 7.8561 | 0.0539 |

In [6]:
geometry_smoke = next(result for result in smoke_results if result.method == GEOMETRY_AWARE_METHOD)
reference_mask = np.asarray(geometry_smoke.reference_result.primal_active_mask, dtype=bool)
archive_masks = np.asarray([entry.primal_active_mask for entry in geometry_smoke.elite_archive], dtype=bool)
if archive_masks.size:
    frequencies = archive_masks.mean(axis=0)
    top_order = np.argsort(-frequencies)[: min(8, frequencies.size)]
    support_rows = [[int(index), fmt(frequencies[index]), str(bool(reference_mask[index]))] for index in top_order]
    display(markdown_table(["constraint", "elite active frequency", "HiGHS active"], support_rows))
else:
    display(Markdown("No elite archive was produced in the smoke check."))

| constraint | elite active frequency | HiGHS active |
| --- | --- | --- |
| 7 | 0.2900 | True |
| 9 | 0.2500 | True |
| 3 | 0.2000 | False |
| 10 | 0.1900 | True |
| 6 | 0.1800 | True |
| 1 | 0.1000 | False |
| 8 | 0.0500 | False |
| 4 | 0.0400 | False |

Recurring support matters because the target product of this method is a ranked set of likely active constraints. A single high-scoring sample is weaker evidence than repeated support among elites.

## Checked-In Artifact Inventory

The notebook now loads every relevant checked-in artifact it can find. Missing files are reported as missing evidence rather than silently filled with assumptions.

In [7]:
ARTIFACTS = {
    "random_dense": [
        Path("outputs/benchmarks/random_dense/results.json"),
    ],
    "solver_hints": [
        Path("outputs/benchmarks/solver_hints/results.json"),
    ],
    "scaling": [
        Path("notebooks/method_viability_outputs/results/scaling/scaling_results.json"),
        Path("results/scaling/scaling_results.json"),
        Path("experiments/results/scaling/scaling_results.json"),
    ],
    "scaling_summary": [
        Path("notebooks/method_viability_outputs/results/scaling/scaling_summary_by_dimension.json"),
        Path("results/scaling/scaling_summary_by_dimension.json"),
        Path("experiments/results/scaling/scaling_summary_by_dimension.json"),
    ],
    "corner_discovery": [
        Path("notebooks/method_viability_outputs/results/corner_discovery/corner_discovery_results.json"),
        Path("results/corner_discovery/corner_discovery_results.json"),
        Path("experiments/results/corner_discovery/corner_discovery_results.json"),
    ],
    "solver_seeding": [
        Path("notebooks/method_viability_outputs/results/solver_seeding/solver_seeding_total_time_results.json"),
        Path("results/solver_seeding/solver_seeding_total_time_results.json"),
        Path("benchmarks/results/solver_seeding/solver_seeding_total_time_results.json"),
    ],
    "certificates": [
        Path("notebooks/method_viability_outputs/results/certificates/certificate_separation_examples.json"),
        Path("results/certificates/certificate_separation_examples.json"),
        Path("experiments/results/certificates/certificate_separation_examples.json"),
    ],
}

loaded: dict[str, tuple[Path | None, dict[str, Any] | None]] = {}
inventory_rows = []
for name, paths in ARTIFACTS.items():
    path, payload = load_json_first(paths)
    loaded[name] = (path, payload)
    records = records_from_payload(payload)
    inventory_rows.append([name, "present" if path else "missing", "-" if path is None else str(path.relative_to(ROOT)), len(records)])

display(markdown_table(["artifact", "status", "path", "records"], inventory_rows))

| artifact | status | path | records |
| --- | --- | --- | --- |
| random_dense | present | outputs/benchmarks/random_dense/results.json | 0 |
| solver_hints | present | outputs/benchmarks/solver_hints/results.json | 0 |
| scaling | missing | - | 0 |
| scaling_summary | missing | - | 0 |
| corner_discovery | present | experiments/results/corner_discovery/corner_discovery_results.json | 40 |
| solver_seeding | present | benchmarks/results/solver_seeding/solver_seeding_total_time_results.json | 144 |
| certificates | missing | - | 0 |

## Active-Set Recovery Versus Naive Sampling

The legacy random-dense benchmark compares geometry-aware sampling and naive Monte Carlo under matched budgets. This is the central structure-discovery evidence.

In [8]:
random_dense_records = records_from_payload(loaded["random_dense"][1])
random_dense_metrics = [
    "active_set_recovery_accuracy",
    "time_to_identify_optimal_active_constraints",
    "objective_gap_to_highs",
    "best_primal_violation",
    "best_dual_violation",
    "best_complementarity_error",
    "wall_clock_seconds",
]
if random_dense_records:
    display(markdown_table(["method", "count", *random_dense_metrics], summarize_by_method(random_dense_records, random_dense_metrics)))
else:
    display(Markdown("No random-dense benchmark artifact is available."))

No random-dense benchmark artifact is available.

In [9]:
if random_dense_records:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, metric, title in [
        (axes[0], "active_set_recovery_accuracy", "Active-set recovery"),
        (axes[1], "objective_gap_to_highs", "Objective gap to HiGHS"),
    ]:
        grouped = defaultdict(list)
        for row in random_dense_records:
            value = safe_float(row.get(metric))
            if value is not None:
                grouped[METHOD_LABELS.get(str(row.get("method")), str(row.get("method")))].append(value)
        labels = list(grouped)
        values = [float(np.mean(grouped[label])) for label in labels]
        ax.bar(labels, values)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    plt.show()

## Solver-Hint Evidence

The solver-hint experiment asks whether the sampled support contains the reference active constraints, even when exact reconstruction fails. This is directly aligned with the method's intended role.

In [10]:
solver_hint_records = records_from_payload(loaded["solver_hints"][1])
solver_hint_metrics = [
    "hint_active_set_jaccard",
    "constraints_in_top_k_support",
    "reconstruction_success",
    "objective_gap_to_highs",
    "wall_clock_seconds",
]
if solver_hint_records:
    display(markdown_table(["method", "count", *solver_hint_metrics], summarize_by_method(solver_hint_records, solver_hint_metrics)))
else:
    display(Markdown("No solver-hint artifact is available."))

No solver-hint artifact is available.

## Vertex Reconstruction And Polishing Limits

Support recovery is not the same as solving the LP. The reconstructed vertices must be feasible, numerically valid, and competitive with the HiGHS reference before they can be treated as candidate solutions.

In [11]:
corner_records = records_from_payload(loaded["corner_discovery"][1])
corner_metrics = [
    "active_set_jaccard_to_scipy_optimum",
    "true_active_containment",
    "reconstruction_success_rate",
    "fraction_feasible_reconstructed_vertices",
    "num_unique_reconstructed_vertices",
]
if corner_records:
    grouped_rows = []
    by_n = defaultdict(list)
    for row in corner_records:
        by_n[int(row.get("n", -1))].append(row)
    for n, rows in sorted(by_n.items()):
        grouped_rows.append([n, len(rows), *[fmt(mean([row.get(metric) for row in rows])) for metric in corner_metrics]])
    display(markdown_table(["n", "count", *corner_metrics], grouped_rows))
else:
    display(Markdown("No corner-discovery artifact is available."))

| n | count | active_set_jaccard_to_scipy_optimum | true_active_containment | reconstruction_success_rate | fraction_feasible_reconstructed_vertices | num_unique_reconstructed_vertices |
| --- | --- | --- | --- | --- | --- | --- |
| 2 | 10 | 1.0000 | 1.0000 | 1.0000 | 0.3812 | 14.8000 |
| 5 | 10 | 0.9667 | 0.9800 | 1.0000 | 0.1798 | 201.8000 |
| 10 | 10 | 0.6286 | 0.7400 | 1.0000 | 0.0597 | 556.3000 |
| 20 | 10 | 0.3862 | 0.5350 | 0.5000 | 0.0016 | 623.4000 |

## Scaling By Dimension And Constraint Ratio

Scaling artifacts are treated as the main stress test. A useful structure-discovery method should retain active-set containment longer than it retains certified solution quality.

In [12]:
scaling_records = records_from_payload(loaded["scaling"][1])
scaling_summary_records = records_from_payload(loaded["scaling_summary"][1])
if scaling_summary_records:
    rows = []
    for row in scaling_summary_records:
        rows.append([
            row.get("n"),
            row.get("m_ratio"),
            row.get("count"),
            fmt(row.get("mean_success_rate")),
            fmt(row.get("mean_active_set_jaccard")),
            fmt(row.get("mean_true_active_containment")),
            fmt(row.get("mean_relative_error_best_feasible")),
            fmt(row.get("failure_rate_no_feasible_vertex")),
            fmt(row.get("failure_rate_infeasible_high_objective")),
        ])
    display(markdown_table(["n", "m/n", "count", "recon success", "active Jaccard", "containment", "rel error", "no feasible rate", "infeasible artifact rate"], rows))
elif scaling_records:
    display(markdown_table(["method", "count", "relative_error", "active_set_jaccard", "true_active_containment", "reconstruction_success", "certificate_success"], summarize_by_method(scaling_records, ["relative_error", "active_set_jaccard", "true_active_containment", "reconstruction_success", "certificate_success"])))
else:
    display(Markdown("No scaling artifact is available."))

No scaling artifact is available.

## Certificate Separation

Raw gaps from arbitrary sampled primal-dual pairs are diagnostics. A certified gap requires both a primal-feasible lower bound and a dual-feasible upper bound.

In [13]:
certificate_payload = loaded["certificates"][1]
if certificate_payload:
    diagnostic = certificate_payload.get("raw_diagnostic_only", {})
    certified = certificate_payload.get("valid_certificate", {})
    display(markdown_table(
        ["case", "raw gap", "certified gap", "lower bound", "upper bound", "valid certificate"],
        [
            ["raw diagnostic", fmt(diagnostic.get("raw_gap")), fmt(diagnostic.get("certified_gap")), fmt(diagnostic.get("best_feasible_primal_lower_bound")), fmt(diagnostic.get("best_feasible_dual_upper_bound")), str(diagnostic.get("has_valid_certificate"))],
            ["valid certificate", fmt(certified.get("raw_gap")), fmt(certified.get("certified_gap")), fmt(certified.get("best_feasible_primal_lower_bound")), fmt(certified.get("best_feasible_dual_upper_bound")), str(certified.get("has_valid_certificate"))],
        ],
    ))
else:
    display(Markdown("No certificate-separation artifact is available. Run `python experiments/run_full_extension_evaluation.py --stage certificates --output-dir .` or enable `RUN_FULL_BENCHMARKS`."))

No certificate-separation artifact is available. Run `python experiments/run_full_extension_evaluation.py --stage certificates --output-dir .` or enable `RUN_FULL_BENCHMARKS`.

## Total-Time And Solver-Use Caveat

Even if active-set hints are useful, the method is only a solver accelerator if sampling, polishing, seed construction, and solve time together beat the baseline. Otherwise it remains a diagnostic or hint generator.

In [14]:
seeding_records = records_from_payload(loaded["solver_seeding"][1])
if seeding_records:
    display(markdown_table(
        ["method", "count", "total_time_seconds", "solver_time_seconds", "speedup_solver_only", "speedup_total", "relative_error"],
        summarize_by_method(seeding_records, ["total_time_seconds", "solver_time_seconds", "speedup_solver_only", "speedup_total", "relative_error"], method_key="method_name"),
    ))
else:
    display(Markdown("No solver-seeding total-time artifact is available. Without it, the notebook cannot claim solver acceleration."))

| method | count | total_time_seconds | solver_time_seconds | speedup_solver_only | speedup_total | relative_error |
| --- | --- | --- | --- | --- | --- | --- |
| sampler_plus_polishing | 18 | 0.1425 | 0.0000 | - | 0.0055 | 0.0022 |
| sampler_plus_vertex_polishing | 18 | 0.1576 | 0.0000 | - | 0.0050 | 3.682e-17 |
| sampler_raw_only | 18 | 0.1424 | 0.0000 | - | 0.0055 | 0.0271 |
| sampler_seed_diagnostics_plus_scipy_highs | 18 | 0.1432 | 7.631e-04 | 1.0000 | 0.0055 | 0.0000 |
| scipy_highs_only | 18 | 7.631e-04 | 7.631e-04 | 1.0000 | 1.0000 | 0.0000 |
| seeded_educational_simplex_from_polished_vertex | 18 | 0.1589 | 0.0013 | 0.6104 | 0.0030 | 3.682e-17 |
| seeded_educational_simplex_from_sampled_active_set | 18 | 0.1434 | 0.0010 | 0.9858 | 0.0033 | 2.799e-17 |
| unseeded_educational_simplex | 18 | 4.372e-04 | 4.372e-04 | 1.0000 | 1.0000 | 6.400e-17 |

## Viability Verdict

The final decision below is computed from the available evidence. It deliberately separates structure discovery from solver replacement.

In [15]:
def method_mean(records: list[dict[str, Any]], method: str, metric: str, method_key: str = "method") -> float | None:
    return mean([row.get(metric) for row in records if str(row.get(method_key)) == method])

geom_active = method_mean(random_dense_records, GEOMETRY_AWARE_METHOD, "active_set_recovery_accuracy")
naive_active = method_mean(random_dense_records, NAIVE_MONTE_CARLO_METHOD, "active_set_recovery_accuracy")
geom_topk = method_mean(solver_hint_records, GEOMETRY_AWARE_METHOD, "constraints_in_top_k_support")
naive_topk = method_mean(solver_hint_records, NAIVE_MONTE_CARLO_METHOD, "constraints_in_top_k_support")
geom_reconstruction = method_mean(solver_hint_records, GEOMETRY_AWARE_METHOD, "reconstruction_success")

available_containment = []
for row in corner_records + scaling_records:
    value = safe_float(row.get("true_active_containment"))
    if value is not None:
        available_containment.append(value)
mean_containment = mean(available_containment)

structure_signals = [
    geom_active is not None and naive_active is not None and geom_active >= naive_active,
    geom_topk is not None and naive_topk is not None and geom_topk >= naive_topk,
    mean_containment is not None and mean_containment >= 0.8,
]
structure_viable = sum(bool(item) for item in structure_signals) >= 2

certificate_available = certificate_payload is not None
seeding_available = bool(seeding_records)
solver_replacement_supported = False
if seeding_available:
    speedups = [safe_float(row.get("speedup_total")) for row in seeding_records]
    speedups = [value for value in speedups if value is not None]
    solver_replacement_supported = bool(speedups and np.mean(speedups) > 1.0 and certificate_available)

lines = [
    "### Computed Verdict",
    "",
    f"- Geometry-aware mean active-set recovery: `{fmt(geom_active)}` versus naive `{fmt(naive_active)}`.",
    f"- Geometry-aware top-k active support containment: `{fmt(geom_topk)}` versus naive `{fmt(naive_topk)}`.",
    f"- Mean available true-active containment from corner/scaling artifacts: `{fmt(mean_containment)}`.",
    f"- Solver-hint reconstruction success in available hint artifact: `{fmt(geom_reconstruction)}`.",
    f"- Certificate-separation artifact present: `{certificate_available}`.",
    f"- Solver total-time artifact present: `{seeding_available}`.",
    "",
    "**Structure-discovery viability:** " + ("supported by the available evidence." if structure_viable else "not established by the available evidence."),
    "",
    "**Solver-replacement viability:** " + ("supported by the available evidence." if solver_replacement_supported else "not supported by this evidence; treat the method as a diagnostic and hint generator."),
]
display(Markdown("\n".join(lines)))

### Computed Verdict

- Geometry-aware mean active-set recovery: `-` versus naive `-`.
- Geometry-aware top-k active support containment: `-` versus naive `-`.
- Mean available true-active containment from corner/scaling artifacts: `0.8137`.
- Solver-hint reconstruction success in available hint artifact: `-`.
- Certificate-separation artifact present: `False`.
- Solver total-time artifact present: `True`.

**Structure-discovery viability:** not established by the available evidence.

**Solver-replacement viability:** not supported by this evidence; treat the method as a diagnostic and hint generator.

## Reproducibility Commands

Default notebook execution should work without long regeneration. To regenerate complete extension artifacts from the repository root, run:

```bash
. .venv/bin/activate
python experiments/run_full_extension_evaluation.py --output-dir .
```

For a faster reduced check:

```bash
. .venv/bin/activate
python experiments/run_full_extension_evaluation.py --quick --output-dir .
```

Recommended validation tests for notebook-supporting code:

```bash
. .venv/bin/activate
pytest tests/test_corner_discovery_schema.py tests/test_scaling_aggregates.py tests/test_superoptimality_scoring.py tests/test_feasibility_first_ranking.py
```